# Step 1 — Setup & Data Partitioning

This notebook prepares **everything** the downstream notebooks need:

1. Installs Python dependencies
2. Mounts Google Drive (so caches survive Colab disconnects)
3. Loads your **positive** and **negative** protein FASTA files
4. Creates a stratified **70% train / 30% independent test** split
5. Creates stratified **5-fold** and **10-fold** CV indices on the training portion
6. Saves everything to `WORK_DIR` so the other notebooks pick it up automatically

**Run cells one at a time, top to bottom.**

You only ever edit two paths (positive FASTA and negative FASTA). The 30% independent test, the 5-fold splits, and the 10-fold splits are all derived automatically — no other input files are needed.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q biopython scikit-learn umap-learn shap matplotlib seaborn pandas
!pip install -q transformers accelerate sentencepiece
import torch
print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Cell 2 — Mount Google Drive

Recommended so cached features and partition files survive between sessions. If you skip this, everything still works but lives in `/content/` and is wiped on runtime reset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3 — Configuration (HARDCODED — edit only these two paths)

Put your two FASTA files anywhere convenient (Drive, `/content/`, etc.) and set the paths below. Nothing else needs changing.

In [ ]:
# ============================================================
# EDIT THESE TWO PATHS — and nothing else
# ============================================================
POSITIVE_FASTA = '/content/drive/MyDrive/pscv_data/positive.fasta'
NEGATIVE_FASTA = '/content/drive/MyDrive/pscv_data/negative.fasta'

# All pipeline artifacts (partitions, cached features, trained models,
# plots, results) go under this directory.
WORK_DIR = '/content/drive/MyDrive/pscv_work'
# ============================================================

# Fixed protocol — do not change unless you know what you're doing
INDEPENDENT_TEST_RATIO = 0.30   # 70/30 train / independent-test
CV_FOLDS = [5, 10]              # 5-fold and 10-fold CV
RANDOM_SEED = 42

import os
os.makedirs(WORK_DIR, exist_ok=True)
print('Work directory   :', WORK_DIR)
print('Positive FASTA   :', POSITIVE_FASTA)
print('Negative FASTA   :', NEGATIVE_FASTA)
print('Indep test ratio :', INDEPENDENT_TEST_RATIO)
print('CV folds         :', CV_FOLDS)
print('Random seed      :', RANDOM_SEED)

## Cell 4 — Load FASTA files

Keeps only sequences containing the 20 standard amino acids. Rare residues (`U`,`Z`,`O`,`B`) get replaced with `X` for tolerance, but anything wildly non-canonical is dropped.

In [ ]:
from Bio import SeqIO
import pandas as pd

STD_AA = set('ACDEFGHIKLMNPQRSTVWY')

def load_fasta(path, label):
    records = []
    n_skipped = 0
    for rec in SeqIO.parse(path, 'fasta'):
        seq = str(rec.seq).upper().strip()
        if not seq:
            n_skipped += 1
            continue
        seq_norm = ''.join(('X' if c in 'UZOB' else c) for c in seq)
        if any(c not in STD_AA and c != 'X' for c in seq_norm):
            n_skipped += 1
            continue
        records.append({
            'id': rec.id,
            'sequence': seq_norm,
            'label': label,
            'length': len(seq_norm),
        })
    print(f'  loaded {len(records)} sequences from {path} (skipped {n_skipped})')
    return records

print('Positive class:')
pos_records = load_fasta(POSITIVE_FASTA, 1)
print('Negative class:')
neg_records = load_fasta(NEGATIVE_FASTA, 0)

df = pd.DataFrame(pos_records + neg_records).reset_index(drop=True)
print()
print('Class distribution:')
print(df['label'].value_counts().rename({1: 'positive', 0: 'negative'}))
print()
print('Sequence length stats per class:')
print(df.groupby('label')['length'].describe().round(1))

## Cell 5 — Create 70/30 stratified train / independent-test split

The independent test set is **never seen during training or cross-validation**. It is the final unbiased estimate of generalization.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, indep_test_df = train_test_split(
    df,
    test_size=INDEPENDENT_TEST_RATIO,
    stratify=df['label'],
    random_state=RANDOM_SEED,
)
train_df = train_df.reset_index(drop=True)
indep_test_df = indep_test_df.reset_index(drop=True)

print(f"Training set       : n={len(train_df):5d}  "
      f"(pos={int((train_df.label==1).sum())}, neg={int((train_df.label==0).sum())})")
print(f"Independent test   : n={len(indep_test_df):5d}  "
      f"(pos={int((indep_test_df.label==1).sum())}, neg={int((indep_test_df.label==0).sum())})")

train_df.to_csv(os.path.join(WORK_DIR, 'train.csv'), index=False)
indep_test_df.to_csv(os.path.join(WORK_DIR, 'independent_test.csv'), index=False)
print()
print('Saved:')
print(' ', os.path.join(WORK_DIR, 'train.csv'))
print(' ', os.path.join(WORK_DIR, 'independent_test.csv'))

## Cell 6 — Build stratified 5-fold and 10-fold CV indices

CV folds are generated **on the training set only** (the 70%). The independent test set is held out until final evaluation. The fold indices are saved as JSON so every downstream notebook uses identical splits.

In [ ]:
from sklearn.model_selection import StratifiedKFold
import json
import numpy as np

y_train = train_df['label'].values

def build_fold_indices(n_splits):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    folds = []
    for fi, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_train)), y_train)):
        folds.append({
            'fold': fi,
            'train_idx': tr_idx.tolist(),
            'val_idx':   va_idx.tolist(),
        })
    return folds

for k in CV_FOLDS:
    folds = build_fold_indices(k)
    out_path = os.path.join(WORK_DIR, f'folds_{k}.json')
    with open(out_path, 'w') as f:
        json.dump(folds, f)
    print(f'\n{k}-fold CV splits:')
    for fold in folds:
        tr_lbl = y_train[fold['train_idx']]
        va_lbl = y_train[fold['val_idx']]
        print(f"  fold {fold['fold']}: "
              f"train={len(tr_lbl):4d} (pos={int(tr_lbl.sum()):3d})  "
              f"val={len(va_lbl):4d} (pos={int(va_lbl.sum()):3d})")
    print(f'  saved -> {out_path}')

## Cell 7 — Verify and summarize

In [ ]:
print('Files in', WORK_DIR, ':')
for f in sorted(os.listdir(WORK_DIR)):
    full = os.path.join(WORK_DIR, f)
    if os.path.isfile(full):
        size_kb = os.path.getsize(full) / 1024
        print(f'  {f:30s}  {size_kb:8.1f} KB')

print()
print('Step 1 complete. Open Step 2 (feature extraction) next.')